In [1]:
import torch
from data_processing_torch import *
from notebooks_creation_models.VGG_Arthur import *
import pandas as pd
import numpy as np
import pickle
import os
from deel import torchlip
# import pdb
import yaml

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
with open('./notebooks_creation_models/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
_, test_loader = load_cifar10(cfg)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [4]:
import sys
sys.path.append("..")
import liresnet.models as models

In [5]:
print("loading model :")
model1 = load_model().vanilla_export().to(device)
model1.load_state_dict(torch.load('/home/aws_install/robustess_project/lip_notebooks/notebooks_creation_models/Vgg_lip_multisteplr_van.pt', weights_only=True))
model1.eval()

loading model :


/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): ParametrizationList(
        (0): _SpectralNorm()
        (1): _BjorckNorm()
        (2): _LConvNorm()
      )
    )
  )
  (norm): BatchCentering()
  (activation): GroupSort2()
  (scalar): MultiplyByScalar()
)
  warnings.warn(
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): Pa

Sequential(
  (0): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): GroupSort2()
    (scalar): MultiplyByScalar()
  )
  (1): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): GroupSort2()
    (scalar): MultiplyByScalar()
  )
  (2): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), p

In [6]:
weights = torch.load('/home/aws_install/robustess_project/lip_models/cifar10-12x512_799.pth').get('backbone')
with open('/home/aws_install/robustess_project/liresnet/configs/cifar10.yaml', 'r') as f:
        cfg = yaml.load(f, Loader=yaml.Loader)
model_cfg = cfg['model']
dataset_cfg = cfg['dataset']
gloro_cfg = cfg['gloro']
model2 = models.GloroNet(**model_cfg, **dataset_cfg)
model2.load_state_dict(weights)
model2 = model2.to(device)
model2.eval()


GloroNet(
  (stem): Sequential(
    (0): Conv2d(3, 512, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), output_padding=(1, 1))
    (1): MinMax(dim=1)
  )
  (conv): LiResConv(
    depth=12, width=512, centering=True
    (act): MinMax(dim=1)
  )
  (neck): Map2Vec(
    (activation): MinMax(dim=1)
  )
  (linear): LiResMLP(
    depth=8, width=2048
    (act): MinMax(dim=1)
  )
  (head): head(in_features=2048, out_features=10, bias=True)
)

In [7]:
images, targets = select_data_for_radius_evaluation_saved_points(test_loader, test_loader.dataset, model1, model2, schedulefree=False)

In [8]:
torch.argmax(model1(images.to(device)), dim=1) == torch.argmax(model2(images.to(device)), dim=1)

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, Tr

In [9]:
targets

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7,
        7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8,
        8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9,
        9, 9, 9, 9, 9, 9, 9, 9])

In [10]:
# 2. Define the directory and create it if it doesn't exist
output_dir = "./benchmark_dataset"
os.makedirs(output_dir, exist_ok=True)

# 3. Define the full file paths
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")

# 4. Save the tensors using pickle 
print(f"Saving data to {output_dir}...")
# Save the images tensor
with open(images_path, 'wb') as f:
    pickle.dump(images, f)

# Save the targets tensor
with open(targets_path, 'wb') as f:
    pickle.dump(targets, f)


Saving data to ./benchmark_dataset...


In [20]:
# Define the directory and file paths
output_dir = "./benchmark_dataset"
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")

# --- Load the Tensors ---
print(f"Loading data from {output_dir}...")

# Load the images tensor
with open(images_path, 'rb') as f:
    loaded_images = pickle.load(f)

# Load the targets tensor
with open(targets_path, 'rb') as f:
    loaded_targets = pickle.load(f)

Loading data from ./benchmark_dataset...


In [22]:
loaded_targets==targets

tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, Tr